In [9]:
%run nb_transactional_shared_functions

StatementMeta(, 73ff9579-01f5-49cc-a5db-e1b282799520, 16, Finished, Available, Finished, True)

In [10]:
import re
import pyspark.sql.functions as F

StatementMeta(, 73ff9579-01f5-49cc-a5db-e1b282799520, 17, Finished, Available, Finished, False)

In [11]:
source_schema = "base_transactional"
source_table = "efi"

target_schema = "curated_transactional"
target_table = "efi"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_schema}")

StatementMeta(, 73ff9579-01f5-49cc-a5db-e1b282799520, 18, Finished, Available, Finished, False)

DataFrame[]

In [12]:
location_path = f"Files/curated/transactional/{target_table}"

tbl_properties = """
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite'     = 'true',
    'delta.autoOptimize.autoCompact'       = 'true',
    'delta.logRetentionDuration'           = 'interval 30 days',
    'delta.deletedFileRetentionDuration'   = 'interval 7 days'
)
"""

StatementMeta(, 73ff9579-01f5-49cc-a5db-e1b282799520, 19, Finished, Available, Finished, False)

In [13]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {target_schema}.{target_table} (
    efi_key                             BIGINT,
    unit                                STRING,
    company_key                         STRING,
    member_firm_code                    STRING,
    profit_center_code                  STRING,
    cost_center_code                    STRING,
    gl_account_code                     STRING,
    fe_person_number                    STRING,
    fe_practice_group_key               STRING,
    fe_office_key                       STRING,
    fe_team_key                         STRING,
    lead_partner_person_number          STRING,
    lead_partner_practice_group_key     STRING,
    lead_partner_office_key             STRING,
    lead_partner_team_key               STRING,
    matter_number                       STRING,
    vendor_code                         STRING,
    document_text                       STRING,
    document_date                       DATE, --to review
    document_number                     STRING,
    transaction_document_type_key       STRING,
    posting_period                      STRING,
    posting_date                        DATE, --to review
    document_currency_code              STRING,
    actual_amount_dc                    DOUBLE,
    office_currency_code                STRING,
    actual_amount_oc                    DOUBLE,
    group_currency_code                 STRING,
    actual_amount_gc                    DOUBLE,
    budget_amount_gc                    DOUBLE,
    forecast_amount_gc                  DOUBLE,
    last_updated_date_utc_at_source     TIMESTAMP,

--metadata
    _ingest_date                        DATE,
    _source_file                        STRING,
    _source_system                      STRING,
    _table                              STRING,
    _lineage_id                         STRING,
    _effective_start_datetime           TIMESTAMP,
    _effective_end_datetime             TIMESTAMP,
    _current_flag                       BOOLEAN,
    hashed_pk                           STRING
)
USING DELTA
LOCATION '{location_path}'
{tbl_properties}
""")

StatementMeta(, 73ff9579-01f5-49cc-a5db-e1b282799520, 20, Finished, Available, Finished, False)

DataFrame[]

In [14]:
df = spark.sql(f"""
SELECT DISTINCT
    CAST(-1 AS BIGINT) AS efi_key,
    COALESCE(INITCAP(LOWER(CAST(unit AS STRING))), 'Unknown') AS unit,
    CAST(company_code AS STRING) AS company_key,
    COALESCE(CAST(company_region_src AS STRING), 'Unknown') AS member_firm_code,
    COALESCE(CAST(profit_center_code AS STRING), 'Unknown') AS profit_center_code,
    COALESCE(CAST(cost_center_code AS STRING), 'Unknown') AS cost_center_code,
    COALESCE(CAST(gl_account_code AS STRING), 'Unknown') AS gl_account_code,
    COALESCE(CAST(fe_code AS STRING), 'Unknown') AS fe_person_number,
    CAST(REPLACE(fe_practice_group_code,'#', 'Unknown') AS STRING) AS fe_practice_group_key,
    CAST(fe_office_code AS STRING) AS fe_office_key,
    CAST(fe_team_code AS STRING) AS fe_team_key,

-- Replace # to Unknown
    COALESCE(CAST(CASE WHEN lead_partner_code = '0' THEN 'Unknown' END AS STRING), 'Unknown') AS lead_partner_person_number,
    COALESCE(CAST(CASE WHEN lead_partner_practice_group_code = '#' THEN 'Unknown' ELSE lead_partner_practice_group_code END AS STRING), 'Unknown') AS lead_partner_practice_group_key,
    COALESCE(CAST(CASE WHEN lead_partner_office_code = '#' THEN 'Unknown' ELSE lead_partner_office_code END AS STRING), 'Unknown') AS lead_partner_office_key,
    COALESCE(CAST(CASE WHEN lead_partner_team_code = '#' THEN 'Unknown' ELSE lead_partner_team_code END AS STRING), 'Unknown') AS lead_partner_team_key,
    COALESCE(CAST(CASE WHEN matter_code = '#' THEN 'Unknown' ELSE matter_code END AS STRING), 'Unknown') AS matter_number,
    COALESCE(CAST(CASE WHEN vendor_code = '#' THEN 'Unknown' ELSE vendor_code END AS STRING), 'Unknown') AS vendor_code,
    COALESCE(CAST(CASE WHEN document_text IN ('#',' ') THEN 'Unknown' ELSE document_text END AS STRING), 'Unknown') AS document_text,
    
    COALESCE(TO_DATE(document_date, 'yyyyMMdd'), '1900-01-01') AS document_date,
    COALESCE(CAST(document_number AS STRING), 'Unknown') AS document_number,
    CAST(document_type_code AS STRING) AS transaction_document_type_key,
    COALESCE(CAST(posting_period AS STRING), 'Unknown') AS posting_period,
    COALESCE(TO_DATE(posting_date, 'yyyyMMdd'), '1900-01-01') AS posting_date,
    COALESCE(CAST(document_currency_code AS STRING), 'Unknown') AS document_currency_code,
    CAST(actual_amount_dc AS DOUBLE) AS actual_amount_dc,
    COALESCE(CAST(office_currency_code AS STRING), 'Unknown') AS office_currency_code,
    CAST(actual_amount_oc AS DOUBLE) AS actual_amount_oc,
    COALESCE(CAST(group_currency_code AS STRING), 'Unknown') AS group_currency_code,
    CAST(actual_amount_gc AS DOUBLE) AS actual_amount_gc,
    CAST(budget_amount_gc AS DOUBLE) AS budget_amount_gc,
    CAST(forecast_amount_gc AS DOUBLE) AS forecast_amount_gc,
    TO_TIMESTAMP(last_update_dt, 'dd/MM/yyyy') AS last_updated_date_utc_at_source,

--metadata
    _ingest_date,
    CAST(_source_file AS STRING) AS _source_file,
    CAST(_source_system AS STRING) AS _source_system,
    'efi' AS _table,
    CAST("44444444-4444-4444-aaaa-aaaaaaaaaaaa" AS STRING) AS _lineage_id,
    CURRENT_TIMESTAMP() AS _effective_start_datetime,
    TO_TIMESTAMP('9999-12-31 23:59:59') AS _effective_end_datetime,
    TRUE AS _current_flag,

--hashkey
    md5(concat_ws('|',
        coalesce(company_code,''),
        coalesce(cost_center_code,''),
        coalesce(gl_account_code,''),
        coalesce(fe_code,''),
        coalesce(fe_practice_group_code,''),
        coalesce(fe_office_code,''),
        coalesce(fe_team_code,''),
        coalesce(lead_partner_code,''),
        coalesce(lead_partner_practice_group_code,''),
        coalesce(lead_partner_office_code,''),
        coalesce(lead_partner_team_code,''),
        coalesce(matter_code,''),
        coalesce(document_type_code,''),
        coalesce(document_currency_code,''),
        coalesce(document_number, ''),
        coalesce(office_currency_code,''),
        coalesce(group_currency_code,''),
        coalesce(posting_period,''),
        coalesce(budget_amount_gc,''),
        coalesce(actual_amount_dc,'')
    )) AS hashed_pk
FROM {source_schema}.{source_table}
""")

display(df)


StatementMeta(, 73ff9579-01f5-49cc-a5db-e1b282799520, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8d8b798a-4334-45ea-9ee6-b6832a893018)

In [15]:
validate_duplicates(df, "hashed_pk", 10)

StatementMeta(, 73ff9579-01f5-49cc-a5db-e1b282799520, 22, Finished, Available, Finished, False)

Total duplicate hashed_pk groups: 0


SynapseWidget(Synapse.DataFrame, 3804feb2-a4b1-4a73-93a0-dc9d4b1f2b13)

In [16]:
merge_to_target(df, target_schema, target_table, location_path, "hashed_pk")

StatementMeta(, 73ff9579-01f5-49cc-a5db-e1b282799520, 23, Finished, Available, Finished, True)

Merged into curated_transactional.efi


In [17]:
df = spark.sql(f"""
--SELECT hashed_pk, count(*) FROM curated_transactional.efi 
--group by hashed_pk
SELECT COUNT(*) FROM {target_schema}.{target_table} 
--where hashed_pk ='e48b1c5f8c7495227188f0ac109f8db4'
""")
display(df)

StatementMeta(, 73ff9579-01f5-49cc-a5db-e1b282799520, 24, Finished, Available, Finished, True)

SynapseWidget(Synapse.DataFrame, 5b6df8d5-d083-4129-8d51-9afa3370dfc6)